# 🐼 第5课：pandas & numpy — 量化交易的核心武器

---

> ⭐ **这是整个课程中最重要的一课**，几乎所有量化代码都建立在 pandas 之上

### 🎯 学完这课你能：
- 用 pandas 处理表格数据（K线、交易记录）
- 用 numpy 做高效数学计算
- 计算移动平均线、收益率
- 生成交易信号

### 📌 对比
```
用 Python 列表处理 10 万条 K 线 → 慢，代码多 😫
用 pandas/numpy 处理 10 万条 K 线 → 快 100 倍，代码少 😎
```

In [ ]:
import numpy as np
import pandas as pd
print(f"numpy  版本: {np.__version__}")
print(f"pandas 版本: {pd.__version__}")
print("✅ 导入成功！")

---
## Part 1: numpy — 快速数值计算

In [ ]:
# 创建价格数组
prices = np.array([100, 102, 99, 105, 103, 107, 106, 110])
print(f"价格: {prices}")

# 向量化运算 —— 不需要 for 循环！
print(f"\n全部涨 10%: {prices * 1.1}")
print(f"全部减 5:   {prices - 5}")

# 一行算出每日收益率
returns = np.diff(prices) / prices[:-1] * 100
print(f"\n每日收益率: {np.round(returns, 2)}%")

In [ ]:
# 统计函数（量化核心）
print("📊 统计")
print(f"  均值: {np.mean(prices):.2f}")
print(f"  标准差: {np.std(prices):.2f}  ← 波动率的基础")
print(f"  最大: {np.max(prices)}")
print(f"  最小: {np.min(prices)}")

# 布尔索引 —— 快速筛选
print(f"\n价格 > 105 的: {prices[prices > 105]}")
print(f"上涨日收益率: {returns[returns > 0].round(2)}")

---
## Part 2: pandas DataFrame — 量化交易的瑞士军刀

DataFrame = **表格**，就像 Excel 一样，但能用代码操作。

In [ ]:
# 创建 K 线数据表
data = {
    "date":   ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"],
    "open":   [100.0, 103.0, 107.0, 105.0, 108.0],
    "high":   [105.0, 108.0, 110.0, 109.0, 112.0],
    "low":    [98.0,  102.0, 104.0, 103.0, 107.0],
    "close":  [103.0, 107.0, 105.0, 108.0, 111.0],
    "volume": [1000000, 1200000, 900000, 1100000, 1500000],
}

df = pd.DataFrame(data)
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")

df  # Jupyter 会自动美化显示

In [ ]:
# 添加计算列 —— 量化常用操作
df["daily_return"] = df["close"].pct_change() * 100      # 每日收益率
df["cum_return"] = (1 + df["close"].pct_change()).cumprod() - 1  # 累计收益
df["amplitude"] = (df["high"] - df["low"]) / df["close"] * 100   # 振幅

df.round(2)

In [ ]:
# 筛选数据
print("📈 上涨的日子:")
display(df[df["daily_return"] > 0][["close", "daily_return"]].round(2))

print("\n📊 放量日 (成交量 > 100万 且 收盘价 > 105):")
display(df[(df["volume"] > 1_000_000) & (df["close"] > 105)][["close", "volume"]])

## ⭐ 移动窗口计算 — 量化核心操作

`rolling(window)` 是量化交易中使用频率最高的函数之一。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC']
matplotlib.rcParams['axes.unicode_minus'] = False

# 生成 30 天模拟数据
np.random.seed(42)
dates = pd.date_range("2024-01-01", periods=60, freq="D")
close = 100 + np.cumsum(np.random.randn(60) * 2)
df2 = pd.DataFrame({"close": close}, index=dates)

# 计算均线
df2["MA5"] = df2["close"].rolling(5).mean()
df2["MA20"] = df2["close"].rolling(20).mean()

# 画图
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df2.index, df2["close"], label="收盘价", color="steelblue", linewidth=1.5)
ax.plot(df2.index, df2["MA5"], label="MA5", color="orange", linewidth=1, linestyle="--")
ax.plot(df2.index, df2["MA20"], label="MA20", color="red", linewidth=1, linestyle="--")
ax.set_title("收盘价 + 移动平均线", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ⭐ 用 pandas 生成交易信号

In [ ]:
# 金叉死叉信号
df2["signal"] = 0
df2.loc[df2["MA5"] > df2["MA20"], "signal"] = 1     # 金叉区间
df2.loc[df2["MA5"] <= df2["MA20"], "signal"] = -1    # 死叉区间
df2["signal_change"] = df2["signal"].diff()

# 可视化信号
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df2.index, df2["close"], color="steelblue", linewidth=1.5, label="收盘价")
ax.plot(df2.index, df2["MA5"], color="orange", linewidth=1, linestyle="--", label="MA5")
ax.plot(df2.index, df2["MA20"], color="red", linewidth=1, linestyle="--", label="MA20")

# 标记买卖点
buys = df2[df2["signal_change"] == 2]
sells = df2[df2["signal_change"] == -2]

ax.scatter(buys.index, buys["close"], marker="^", color="green", s=150, zorder=5, label="买入(金叉)")
ax.scatter(sells.index, sells["close"], marker="v", color="red", s=150, zorder=5, label="卖出(死叉)")

ax.set_title("双均线交叉策略信号", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 打印信号
for date, row in df2[df2["signal_change"].abs() == 2].dropna().iterrows():
    emoji = "🟢 金叉买入" if row["signal_change"] > 0 else "🔴 死叉卖出"
    print(f"  {date.strftime('%Y-%m-%d')} {emoji}  价格: {row['close']:.2f}")

## 📊 统计描述

In [ ]:
df2["close"].describe().round(2)

---

## ✅ Python 基础课程全部完成！🎉

### 你已经掌握了：
- ✅ 变量和数据类型
- ✅ 条件判断和循环
- ✅ 函数定义和使用
- ✅ 列表、字典等数据结构
- ✅ pandas DataFrame 数据处理
- ✅ numpy 数值计算
- ✅ matplotlib 画图

### ➡️ 下一步
进入 `../01-market-data/` 获取**真实行情数据**！